# ETH, SETH y lower bounds condicionales

Ilustramos los lower bounds condicionales derivados de la Hipotesis Exponencial (ETH) y la Hipotesis Exponencial Fuerte (SETH).

Articulo: `04/13-eth-seth-consecuencias.md`

In [ ]:
import time
import random
import math
from itertools import product

## 1. Subsecuencia comun mas larga (LCS)

El algoritmo clasico de LCS corre en O(n*m). Bajo SETH, no existe un algoritmo O(n^{2-eps}) para ningun eps>0.
Vemos empiricamente el comportamiento cuadratico.

In [ ]:
def lcs(s1, s2):
    n, m = len(s1), len(s2)
    dp = [[0]*(m+1) for _ in range(n+1)]
    for i in range(1, n+1):
        for j in range(1, m+1):
            if s1[i-1] == s2[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[n][m]

# Medir tiempo vs tamano de entrada
alphabet = 'ACGT'
resultados = []
for n in [50, 100, 200, 400, 600]:
    s1 = ''.join(random.choices(alphabet, k=n))
    s2 = ''.join(random.choices(alphabet, k=n))
    t0 = time.perf_counter()
    val = lcs(s1, s2)
    elapsed = time.perf_counter() - t0
    resultados.append((n, elapsed, val))
    print(f"n={n:4d}: LCS={val:4d}, tiempo={elapsed*1000:.2f} ms")

# Verificar que el crecimiento es ~cuadratico
ns = [r[0] for r in resultados]
ts = [r[1] for r in resultados]
ratio = ts[-1] / ts[0]
ratio_esperado = (ns[-1] / ns[0])**2
print(f"\nRatio de tiempo (n={ns[-1]} vs n={ns[0]}): {ratio:.1f}x")
print(f"Ratio esperado cuadratico: {ratio_esperado:.1f}x")

## 2. Edit Distance (distancia de edicion)

Similar a LCS: el algoritmo de Wagner-Fischer es O(n^2). Bajo SETH no se puede mejorar sustancialmente.

In [ ]:
def edit_distance(s1, s2):
    n, m = len(s1), len(s2)
    dp = list(range(m+1))
    for i in range(1, n+1):
        prev = dp[:]
        dp[0] = i
        for j in range(1, m+1):
            if s1[i-1] == s2[j-1]:
                dp[j] = prev[j-1]
            else:
                dp[j] = 1 + min(prev[j], dp[j-1], prev[j-1])
    return dp[m]

# Comparar LCS y Edit Distance
print("Comparacion LCS vs Edit Distance:")
casos = [("ABCBDAB", "BDCAB"), ("AGGTAB", "GXTXAYB"), ("kitten", "sitting")]
for s1, s2 in casos:
    l = lcs(s1, s2)
    d = edit_distance(s1, s2)
    print(f"  '{s1}' vs '{s2}': LCS={l}, EditDist={d}")

## 3. k-SAT: la hipotesis exponencial (ETH)

ETH afirma que 3-SAT no puede resolverse en O(2^{o(n)}) donde n es el numero de variables.
Vemos empiricamente el crecimiento exponencial de la busqueda exhaustiva.

In [ ]:
def sat_fuerza_bruta(formula, n_vars):
    ops = 0
    for bits in range(1 << n_vars):
        ops += 1
        asig = {v+1: bool((bits >> v) & 1) for v in range(n_vars)}
        if all(any((l>0)==asig[abs(l)] for l in c) for c in formula):
            return True, ops
    return False, ops

# Medir tiempo vs numero de variables en instancias aleatorias de 3-SAT
random.seed(0)
print("Crecimiento exponencial de 3-SAT (fuerza bruta):")
for n in range(5, 21):
    # Instancia aleatoria con ratio 4.27 (cerca del umbral de fase)
    m = int(4.27 * n)
    formula = [[random.choice([-1,1])*(random.randint(1,n)) for _ in range(3)] for _ in range(m)]
    t0 = time.perf_counter()
    sat, ops = sat_fuerza_bruta(formula, n)
    elapsed = time.perf_counter() - t0
    print(f"  n={n:2d}: {ops:7d} ops, {elapsed*1000:.1f} ms, sat={sat}")

## 4. Consecuencias concretas de ETH/SETH

Bajo ETH/SETH, los siguientes lower bounds son condicionales:

| Problema | Complejidad conocida | Lower bound (ETH/SETH) |
|----------|---------------------|------------------------|
| LCS de dos cadenas | O(n^2) | Omega(n^{2-eps}) bajo SETH |
| Edit Distance | O(n^2) | Omega(n^{2-eps}) bajo SETH |
| k-SAT | O(2^n) | Omega(2^{delta*n}) bajo ETH |
| 3-Colorabilidad | O(2^n) | Omega(2^{delta*n}) bajo ETH |
| Subgrafo isomorfico k-clique | O(n^k) | Omega(n^{k/4}) bajo ETH |

In [ ]:
print("Resumen: ETH y SETH como hipotesis de complejidad")
print()
print("ETH (Impagliazzo-Paturi 2001):")
print("  3-SAT requiere tiempo 2^{Omega(n)} en el peor caso.")
print()
print("SETH (Impagliazzo-Paturi-Zane 2001):")
print("  Para todo k, k-SAT requiere tiempo 2^{(1-eps_k)*n} con eps_k -> 0.")
print()
print("Consecuencia directa (Williams 2005, Bringmann 2015):")
print("  Cualquier algoritmo para LCS o Edit Distance con tiempo O(n^{2-eps})")
print("  refutaria SETH.")
print()
print("Esto explica por que decadas de investigacion no han mejorado")
print("el algoritmo cuadratico para LCS/Edit Distance: probablemente es optimo.")